# Building MCP-Powered Agentic RAG Application: Step-by-Step Guide (1/2)

Run from this series folder with Python 3.13. The Qdrant default is genuine durable local mode, not a Docker or server simulation. Part 2 retains the original Nomic model and ten complete FAQ topics, with native Transformers code and no remote-code trust. Configure API credentials locally; never paste them into notebooks. Paid model calls require RUN_OPENAI=1. No tracing/upload is enabled.

```bash
python -m pip install mcp==2.3.0 qdrant-client==1.20.0 llama-index-embeddings-huggingface==0.8.0 sentence-transformers==6.2.0 transformers==5.22.1 torch==2.15.0 python-dotenv==1.2.4 requests==2.34.2 ddgs==9.16.1 openai==3.42.1 filelock==4.2.7 beautifulsoup4==4.15.0
```

In [ ]:
import os, json, sys, time, tempfile, subprocess, socket, hashlib
from pathlib import Path
from importlib.metadata import version
import requests
from mcp import Client
from mcp.server import MCPServer
from qdrant_client import QdrantClient, models
project = Path.cwd()
assert (project / 'foundation_server.py').is_file(), 'Run from the supplied series folder.'
work = Path(tempfile.mkdtemp(prefix='tdb-mcp-tutorial-'))
for name in ['mcp','qdrant-client','llama-index-embeddings-huggingface','openai']:
    print(name, version(name))

In [ ]:
HOST, PORT = '127.0.0.1', 8080
MCP_URL = f'http://{HOST}:{PORT}/mcp'
print('Loopback endpoint:', MCP_URL)
print('The MCP server exposes tools; the AI host owns model selection and generation.')

In [ ]:
store = work / 'foundation-qdrant'
database = QdrantClient(path=str(store))
database.create_collection('foundation', vectors_config=models.VectorParams(size=768, distance=models.Distance.COSINE))
assert database.count('foundation', exact=True).count == 0
database.close()
database = QdrantClient(path=str(store))
assert database.get_collection('foundation').config.params.vectors.size == 768
print('Durable local Qdrant collection: 768 dimensions, 0 records, reopened successfully.')
database.close()

In [ ]:
"""Part 1's protocol server: no LLM or tools are implied by construction."""
import os
from mcp.server import MCPServer

HOST = "127.0.0.1"
PORT = int(os.getenv("MCP_PORT", "8080"))
mcp_server = MCPServer("MCP-RAG-app", instructions="Part 1 protocol foundation.")


print('MCPServer constructed. Part 1 intentionally registers no retrieval/search tools yet.')

In [ ]:
def free_port():
    with socket.socket() as sock:
        sock.bind(('127.0.0.1', 0))
        return sock.getsockname()[1]

def launch_server(filename, extra_env=None):
    port = free_port()
    env = dict(os.environ, MCP_PORT=str(port), TOKENIZERS_PARALLELISM='false')
    env.update(extra_env or {})
    # Do not send hosted-provider credentials to a read-only local tool server.
    for key in ['OPENAI_API_KEY','LANGSMITH_API_KEY','HF_TOKEN']:
        env.pop(key, None)
    log = (work / (filename + '.log')).open('w')
    process = subprocess.Popen([sys.executable, str(project / filename)], env=env,
        cwd=project, stdout=log, stderr=subprocess.STDOUT)
    try:
        for _ in range(900):
            if process.poll() is not None:
                raise RuntimeError((work / (filename + '.log')).read_text())
            try:
                response = requests.get(f'http://127.0.0.1:{port}/mcp', timeout=1)
                if response.status_code in [200,400,405,406]:
                    return process, log, f'http://127.0.0.1:{port}/mcp'
            except requests.RequestException:
                pass
            time.sleep(0.1)
        raise RuntimeError('Owned local server startup timed out.')
    except BaseException:
        process.terminate();process.wait(timeout=10);log.close()
        raise

def stop_server(process, log):
    process.terminate()
    try: process.wait(timeout=10)
    except subprocess.TimeoutExpired: process.kill();process.wait(timeout=10)
    log.close()
    assert process.poll() is not None
print('Owned-server startup/shutdown helpers defined.')

In [ ]:
process, log, endpoint = launch_server('foundation_server.py')
try:
    async with Client(endpoint, read_timeout_seconds=30) as session:
        listed = await session.list_tools()
        assert listed.tools == []
        print('Real Streamable HTTP connected; registered tools:', len(listed.tools))
        print('Protocol version:', session.protocol_version)
        for invalid in [{'messages': [{'role':'user','content':'Hello'}]}, {}]:
            rejected = requests.post(endpoint, json=invalid, timeout=10,
                headers={'Accept':'application/json, text/event-stream'})
            assert rejected.status_code >= 400
        print('Chat-shaped JSON is not an MCP request: rejected.')
finally:
    stop_server(process, log)
print('Owned foundation server stopped.')

## Original source-code appendix

Nonexecuting complete original archive blocks. Run maintained cells above.

```text
/mcp-agentic-rag
|-- .env
|-- mcp_server.py          # Our main application and server logic
|-- rag_app.py       # Our RAG-specific code and data
```

```text
import os
from typing import List
import requests

from dotenv import load_dotenv
from mcp.server.fastmcp import FastMCP

# Updated import: Assumes FAQEngine and the FAQ text are in 'rag_app.py'
from rag_app import FAQEngine, PYTHON_FAQ_TEXT


# Load environment variables from .env file
load_dotenv()

# Configuration constants
QDRANT_URL = "http://localhost:6333"
COLLECTION_NAME = "python_faq_collection"  # Using a new collection for the Python data
HOST = "127.0.0.1"
PORT = 8080
```

```text
pip install mcp-server qdrant-client python-dotenv requests
```

```text
# .env file
# Add any API keys or secrets here later
OPENAI_API_KEY="sk-..."
FIRECRAWL_API_KEY = "Your API Key"
```

```text
docker run -p 6333:6333 -p 6334:6334 \
    -v $(pwd)/qdrant_storage:/qdrant/storage \
    qdrant/qdrant
```

```text
# Create an MCP server instance
mcp_server = FastMCP("MCP-RAG-app",
                     host=HOST,
                     port=PORT,
                     timeout=30)
```

